In [2]:
import warnings

import astropy.units as u
import FunctionLib as FL
import inspect
from tqdm import tqdm
import astropy
import wave
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from collections import defaultdict
import re
import scipy
from astropy.io import fits as asfits

mpl.rcParams['font.family'] = 'serif'

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
warnings.filterwarnings("ignore")

DJAv4Catalog = FL.Spectrum_Catalog()
DJAv4Catalog.load_from_pkl(os.path.expanduser(
    './DJAV4.2Catalog.pkl'))
print(DJAv4Catalog.sample_num())

DJAv4Catalog.to_dataframe()

2370


,survey_id_subid,prism_filepath,prism_redshift,determined_redshift,grating_filepaths,grating_redshifts,file_count,available_filters,properties,survey_id,sample_flag,reason_for_exclusion,grating_within_coverage,grating_slitloss_correction
0,snh0pe-v4_4446_102,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,0.2259,0.2259,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': nan, 'g235m-f170lp': nan}",3,"{g140m-f100lp, prism-clear, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3]",{},{}
1,snh0pe-v4_4446_143,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.6318,1.6311,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.6313, 'g235m-f170lp': 1.6309}",3,"{g140m-f100lp, prism-clear, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3]",{},{}
2,snh0pe-v4_4446_285,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,0.4446,0.4462,{'g235m-f170lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g235m-f170lp': 0.4462, 'g140m-f100lp': 0.4462}",3,"{g140m-f100lp, prism-clear, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3]",{},{}
3,snh0pe-v4_4446_29,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.7834,1.77975,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.7799, 'g235m-f170lp': 1.7796}",3,"{g140m-f100lp, prism-clear, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3]",{},{}
4,snh0pe-v4_4446_123,/home/xingyaocai/DJAv4.2/snh0pe-v4/snh0pe-v4_p...,1.7855,1.7855,{'g140m-f100lp': '/home/xingyaocai/DJAv4.2/snh...,"{'g140m-f100lp': 1.7855, 'g235m-f170lp': 1.7851}",3,"{g140m-f100lp, prism-clear, g235m-f170lp}",{'redshift_conflict': False},snh0pe-v4,False,"[redshift_below_3, redshift_below_3]",{},{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43037,glimpse-obs02-v4_9223_9152,None,None,5.5385,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 5.5385},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,[no_prism_spectrum],{},{}
43038,glimpse-obs02-v4_9223_98001,None,None,2.6322,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 2.6322},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,"[redshift_below_3, no_prism_spectrum]",{},{}
43039,glimpse-obs02-v4_9223_47046,None,None,4.3554,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 4.3554},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,[no_prism_spectrum],{},{}
43040,glimpse-obs02-v4_9223_45350,None,None,1.3675,{'g395m-f290lp': '/home/xingyaocai/DJAv4.2/gli...,{'g395m-f290lp': 1.3675},1,{g395m-f290lp},{'redshift_conflict': False},glimpse-obs02-v4,False,"[redshift_below_3, no_prism_spectrum]",{},{}


In [4]:
DJAv4Catalog.add_property_field('lines_fit', {})
DJAv4Catalog.add_property_field('line_ratios', {})


In [5]:
alpha_flux=[]
alpha_flux_err=[]
alpha_slitloss_correction=[]
beta_flux=[]
beta_flux_err=[]
beta_slitloss_correction=[]
redshift_list=[]
survey_id_subid_list=[]
for survey_id_subid, catalog in tqdm(DJAv4Catalog.catalog_iterator()):
    if catalog['sample_flag']!= True:
        continue

    catalog['sample_flag']=False

    redshift=catalog['determined_redshift']


    try:

        for disperser_filter_name, grating_filepath in catalog['grating_within_coverage'].items():
            disperser_name, filter_name=disperser_filter_name.split('-')
            grating_spectrum=FL.Load_Spectrum_From_Fits(grating_filepath, redshift=redshift)

            #alpha

            if len(grating_spectrum.processing_flux_lambda.data)<10:
                continue
            alpharestFrameWavelength=6563.0*astropy.units.AA

            grating_spectrum.set_boundarys(alpharestFrameWavelength-75.0*astropy.units.AA, alpharestFrameWavelength+75.0*astropy.units.AA)


            alphaspectralLineFitter=FL.SpectralLineFitter(grating_spectrum,alpharestFrameWavelength)

            alpha_fit_result=alphaspectralLineFitter.fit_single_gaussian_with_offset()

            alpha_fitting_uncertainty_value, alpha_flux_calculated=FL.fitting_uncertainty(alpha_fit_result)

            SNR=alpha_flux_calculated/alpha_fitting_uncertainty_value

            if SNR<3:
                continue

            #beta

            grating_spectrum.reset()
            betarestFrameWavelength=4861.0*astropy.units.AA
            grating_spectrum.set_boundarys(betarestFrameWavelength-75.0*astropy.units.AA, betarestFrameWavelength+75.0*astropy.units.AA)

            betaspectralLineFitter=FL.SpectralLineFitter(grating_spectrum,betarestFrameWavelength)

            beta_fit_result=betaspectralLineFitter.fit_single_gaussian_with_offset()

            beta_fitting_uncertainty_value, beta_flux_calculated=FL.fitting_uncertainty(beta_fit_result)

            beta_SNR=beta_flux_calculated/beta_fitting_uncertainty_value

            if beta_SNR<1:
                continue

            alpha_flux.append(alpha_flux_calculated)
            alpha_flux_err.append(alpha_fitting_uncertainty_value)
            catalog['lines_fit'][f'{disperser_filter_name}']={'halpha_flux':alpha_flux_calculated,'halpha_flux_err':alpha_fitting_uncertainty_value}



            beta_flux.append(beta_flux_calculated)
            beta_flux_err.append(beta_fitting_uncertainty_value)
            catalog['lines_fit'][f'{disperser_filter_name}'].update({'hbeta_flux':beta_flux_calculated,'hbeta_flux_err':beta_fitting_uncertainty_value})
            redshift_list.append(redshift)
            survey_id_subid_list.append(survey_id_subid)
            catalog['sample_flag']=True
            catalog['line_ratios'][f'{disperser_filter_name}']=alpha_flux_calculated/beta_flux_calculated
            DJAv4Catalog.update_catalog_item(survey_id_subid, catalog)


    except Exception as e:
        print(e)





16830it [00:09, 2893.33it/s]

'covariance'


43042it [00:27, 1587.57it/s]


In [ ]:
alpha_flux=np.array(alpha_flux)
alpha_flux.shape

In [ ]:
count=0
for survey_id_subid, catalog in tqdm(DJAv4Catalog.catalog_iterator()):
    if catalog['sample_flag']!= True:
        continue

    for key in catalog['line_ratios'].keys():
        if catalog['line_ratios'][key]<2.86:
            count+=1


In [6]:
DJAv4Catalog.sample_num()

1246

In [7]:
DJAv4Catalog.save_catalog_to_pkl(os.path.expanduser(
    './DJAV4.2Catalog.pkl'))

In [ ]:

# np.save('./balmer_decrement_npys/alpha_flux.npy', np.array(alpha_flux))
# np.save('./balmer_decrement_npys/alpha_flux_err.npy', np.array(alpha_flux_err))
# np.save('./balmer_decrement_npys/alpha_slitloss_correction.npy', np.array(alpha_slitloss_correction))
# np.save('./balmer_decrement_npys/beta_flux.npy', np.array(beta_flux))
# np.save('./balmer_decrement_npys/beta_flux_err.npy', np.array(beta_flux_err))
# np.save('./balmer_decrement_npys/beta_slitloss_correction.npy', np.array(beta_slitloss_correction))
# np.save('./balmer_decrement_npys/redshift_list.npy', np.array(redshift_list))
# np.save('./balmer_decrement_npys/survey_id_subid_list.npy', np.array(survey_id_subid_list))

alpha_flux=np.load('./balmer_decrement_npys/alpha_flux.npy')
alpha_flux_err=np.load('./balmer_decrement_npys/alpha_flux_err.npy')
alpha_slitloss_correction=np.load('./balmer_decrement_npys/alpha_slitloss_correction.npy')
beta_flux=np.load('./balmer_decrement_npys/beta_flux.npy')
beta_flux_err=np.load('./balmer_decrement_npys/beta_flux_err.npy')
beta_slitloss_correction=np.load('./balmer_decrement_npys/beta_slitloss_correction.npy')
redshift_list=np.load('./balmer_decrement_npys/redshift_list.npy')
survey_id_subid_list=np.load('./balmer_decrement_npys/survey_id_subid_list.npy')

In [8]:
ratio=[]
for survey_id_subid in tqdm(survey_id_subid_list):
    catalog=DJAv4Catalog.catalog[survey_id_subid]
    catalog['line_ratios']['halpha_flux']=alpha_flux[np.where(survey_id_subid_list==survey_id_subid)[0][0]]
    catalog['line_ratios']['halpha_flux_err']=alpha_flux_err[np.where(survey_id_subid_list==survey_id_subid)[0][0]]
    catalog['line_ratios']['beta_flux']=beta_flux[np.where(survey_id_subid_list==survey_id_subid)[0][0]]
    catalog['line_ratios']['beta_flux_err']=beta_flux_err[np.where(survey_id_subid_list==survey_id_subid)[0][0]]
    ratio.append(catalog['line_ratios']['halpha_flux']/catalog['line_ratios']['beta_flux'])
    DJAv4Catalog.update_catalog_item(survey_id_subid, catalog)

  0%|          | 0/1342 [00:00<?, ?it/s]


ValueError: Calling nonzero on 0d arrays is not allowed. Use np.atleast_1d(scalar).nonzero() instead. If the context of this error is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.

In [ ]:
DJAv4Catalog.save_catalog_to_pkl(os.path.expanduser(
    './DJAV4.2Catalog.pkl'))

In [ ]:
plt.hist(ratio, bins=50, range=(0,10))

In [ ]:
balmer_decrement=np.array(alpha_flux)/np.array(beta_flux)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


# --- Code Modification Starts Here ---

# Create a figure
fig = plt.figure(figsize=(50, 10))

# Create a GridSpec layout: 1 row, 2 columns.
# The scatter plot will be 4 times wider than the histogram.
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], wspace=0.05)

# Create the subplot for the scatter plot
ax0 = fig.add_subplot(gs[0, 0])

# Create the subplot for the histogram on the right.
# sharey=ax0 ensures the y-axis is aligned with the scatter plot.
ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)

# --- Original Plotting Logic (adapted for ax0) ---

# Store the balmer_decrement values that will be plotted
balmer_decrement_to_plot = []

for i in range(len(alpha_flux)):
    uncertainty = FL.BalmerDecrementUncertainty(alpha_flux[i], alpha_flux_err[i], beta_flux[i], beta_flux_err[i])
    if uncertainty / balmer_decrement[i] > 0.25:
        continue
    if balmer_decrement[i] > 5 or redshift_list[i] < 4 or redshift_list[i] > 7.5:
        continue

    # Append the valid data point to our list for the histogram
    balmer_decrement_to_plot.append(balmer_decrement[i])

    ax0.scatter(redshift_list[i], balmer_decrement[i], s=10, alpha=0.5, c='k')
    ax0.errorbar(redshift_list[i], balmer_decrement[i], yerr=uncertainty, fmt='o', alpha=0.5, c='k')

ax0.set_xlabel('Redshift')
ax0.set_ylabel(r'$\alpha/\beta$')
ax0.set_ylim(1, 5)
ax0.set_xlim(4, 7.5)
ax0.axhline(y=2.86, color='r', linestyle='--', label='Expected Ratio')

# --- New Histogram Plotting Logic (on ax1) ---

# Plot the vertical histogram on the right subplot
# Use the list of balmer_decrement values that were actually plotted
balmer_decrement_to_plot=np.array(balmer_decrement_to_plot)
balmer_decrement_to_plot=balmer_decrement_to_plot[np.where((balmer_decrement_to_plot>1) & (balmer_decrement_to_plot<=5))]
ax1.hist(balmer_decrement_to_plot, bins=20, orientation='horizontal', color='k', alpha=0.7)

# Hide the y-axis labels and ticks on the histogram plot to avoid clutter
ax1.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
# Optionally, hide the x-axis labels/ticks as well if counts are not important
ax1.set_xlabel('Count')
ax1.tick_params(axis='x', labelsize=8)


# Show the final plot
plt.show()

In [ ]:
alpha_flux_corrected=np.array(alpha_flux)/np.array(alpha_slitloss_correction)
beta_flux_corrected=np.array(beta_flux)/np.array(beta_slitloss_correction)
alpha_flux_err_corrected=np.array(alpha_flux_err)/np.array(alpha_slitloss_correction)
beta_flux_err_corrected=np.array(beta_flux_err)/np.array(beta_slitloss_correction)
balmer_decrement_corrected=alpha_flux_corrected/beta_flux_corrected

In [ ]:
plt.hist(balmer_decrement_to_plot, bins=30, range=(0,5), color='k', alpha=0.7)

In [ ]:
plt.hist(alpha_slitloss_correction, bins=50, range=(0,2), color='k', alpha=0.7)
plt.hist(beta_slitloss_correction, bins=50, range=(0,2), color='r', alpha=0.7)
plt.xlim(0.25,1)
plt.title('Flux Fraction through the Slit Distribution with PSF')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


# --- Code Modification Starts Here ---

# Create a figure
fig = plt.figure(figsize=(50, 10))

# Create a GridSpec layout: 1 row, 2 columns.
# The scatter plot will be 4 times wider than the histogram.
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], wspace=0.05)

# Create the subplot for the scatter plot
ax0 = fig.add_subplot(gs[0, 0])

# Create the subplot for the histogram on the right.
# sharey=ax0 ensures the y-axis is aligned with the scatter plot.
ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)

# --- Original Plotting Logic (adapted for ax0) ---

# Store the balmer_decrement values that will be plotted
balmer_decrement_to_plot_corrected = []

for i in range(len(alpha_flux_corrected)):
    uncertainty = FL.BalmerDecrementUncertainty(alpha_flux_corrected[i], alpha_flux_err_corrected[i], beta_flux_corrected[i], beta_flux_err_corrected[i])
    if uncertainty / balmer_decrement_corrected[i] > 0.25:
        continue
    if balmer_decrement_corrected[i] > 5 or redshift_list[i] < 4 or redshift_list[i] > 7.5:
        continue

    # Append the valid data point to our list for the histogram
    balmer_decrement_to_plot_corrected.append(balmer_decrement_corrected[i])

    ax0.scatter(redshift_list[i], balmer_decrement_corrected[i], s=10, alpha=0.5, c='k')
    ax0.errorbar(redshift_list[i], balmer_decrement_corrected[i], yerr=uncertainty, fmt='o', alpha=0.5, c='k')

ax0.set_xlabel('Redshift')
ax0.set_ylabel(r'$\alpha/\beta$')
ax0.set_ylim(1, 5)
ax0.set_xlim(4, 7.5)
ax0.axhline(y=2.86, color='r', linestyle='--', label='Expected Ratio')

# --- New Histogram Plotting Logic (on ax1) ---

# Plot the vertical histogram on the right subplot
# Use the list of balmer_decrement values that were actually plotted
balmer_decrement_to_plot_corrected=np.array(balmer_decrement_to_plot_corrected)
balmer_decrement_to_plot_corrected=balmer_decrement_to_plot_corrected[np.where((balmer_decrement_to_plot_corrected>1) & (balmer_decrement_to_plot_corrected<=5))]
ax1.hist(balmer_decrement_to_plot_corrected, bins=20, orientation='horizontal', color='k', alpha=0.7)

# Hide the y-axis labels and ticks on the histogram plot to avoid clutter
ax1.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
# Optionally, hide the x-axis labels/ticks as well if counts are not important
ax1.set_xlabel('Count')
ax1.tick_params(axis='x', labelsize=8)


# Show the final plot
plt.show()

In [ ]:
import matplotlib.lines as mlines
fig = plt.figure(figsize=(50, 10))

# Create a GridSpec layout: 1 row, 2 columns.
# The scatter plot will be 4 times wider than the histogram.
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], wspace=0.05)

# Create the subplot for the scatter plot
ax0 = fig.add_subplot(gs[0, 0])

# Create the subplot for the histogram on the right.
# sharey=ax0 ensures the y-axis is aligned with the scatter plot.
ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)

# --- Original Plotting Logic (adapted for ax0) ---

# Store the balmer_decrement values that will be plotted
balmer_decrement_to_plot = []

for i in range(len(alpha_flux)):
    uncertainty = FL.BalmerDecrementUncertainty(alpha_flux[i], alpha_flux_err[i], beta_flux[i], beta_flux_err[i])
    if uncertainty / balmer_decrement[i] > 0.25:
        continue
    if balmer_decrement[i] > 5 or redshift_list[i] < 4 or redshift_list[i] > 7.5:
        continue

    # Append the valid data point to our list for the histogram
    balmer_decrement_to_plot.append(balmer_decrement[i])

    ax0.scatter(redshift_list[i], balmer_decrement[i], s=10, alpha=0.5, c='b')
    ax0.errorbar(redshift_list[i], balmer_decrement[i], yerr=uncertainty, fmt='o', alpha=0.5, c='b')

balmer_decrement_to_plot_corrected = []

for i in range(len(alpha_flux_corrected)):
    uncertainty = FL.BalmerDecrementUncertainty(alpha_flux_corrected[i], alpha_flux_err_corrected[i], beta_flux_corrected[i], beta_flux_err_corrected[i])
    if uncertainty / balmer_decrement_corrected[i] > 0.25:
        continue
    if balmer_decrement_corrected[i] > 5 or redshift_list[i] < 4 or redshift_list[i] > 7.5:
        continue

    # Append the valid data point to our list for the histogram
    balmer_decrement_to_plot_corrected.append(balmer_decrement_corrected[i])

    ax0.scatter(redshift_list[i], balmer_decrement_corrected[i], s=10, alpha=0.5, c='g')
    ax0.errorbar(redshift_list[i], balmer_decrement_corrected[i], yerr=uncertainty, fmt='o', alpha=0.5, c='g')


ax0.set_xlabel('Redshift')
ax0.set_ylabel(r'$\alpha/\beta$')
ax0.set_ylim(1, 5)
ax0.set_xlim(4, 7.5)
red_line = ax0.axhline(y=2.86, color='r', linestyle='--', label='Expected Ratio')

# 为蓝色和绿色散点创建代理（proxy）图例项
blue_proxy = mlines.Line2D([], [], color='b', marker='o', linestyle='None',
                          markersize=5, alpha=0.5, label='Uncorrected')
green_proxy = mlines.Line2D([], [], color='g', marker='o', linestyle='None',
                           markersize=5, alpha=0.5, label='Corrected')

# 将代理项和红线一起放入图例中
ax0.legend(handles=[blue_proxy, green_proxy, red_line])
# --- New Histogram Plotting Logic (on ax1) ---

# Plot the vertical histogram on the right subplot
# Use the list of balmer_decrement values that were actually plotted
balmer_decrement_to_plot=np.array(balmer_decrement_to_plot)
balmer_decrement_to_plot=balmer_decrement_to_plot[np.where((balmer_decrement_to_plot>1) & (balmer_decrement_to_plot<=5))]
balmer_decrement_to_plot_corrected=np.array(balmer_decrement_to_plot_corrected)
balmer_decrement_to_plot_corrected=balmer_decrement_to_plot_corrected[np.where((balmer_decrement_to_plot_corrected>1) & (balmer_decrement_to_plot_corrected<=5))]
ax1.hist(balmer_decrement_to_plot, bins=20, orientation='horizontal', color='b', alpha=0.3, label='Uncorrected')
ax1.hist(balmer_decrement_to_plot_corrected, bins=20, orientation='horizontal', color='g', alpha=0.3, label='Corrected')

# Hide the y-axis labels and ticks on the histogram plot to avoid clutter
ax1.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
# Optionally, hide the x-axis labels/ticks as well if counts are not important
ax1.set_xlabel('Count')
ax1.tick_params(axis='x', labelsize=8)

ax1.legend()

plt.title('Balmer Decrement vs Redshift with Histogram')
# Show the final plot
plt.show()

In [ ]:

fig=plt.figure(figsize=(50,10))
gs = gridspec.GridSpec(1, 2, width_ratios=[6, 1], wspace=0.05)

# Create the subplot for the scatter plot
ax0 = fig.add_subplot(gs[0, 0])

# Create the subplot for the histogram on the right.
# sharey=ax0 ensures the y-axis is aligned with the scatter plot.
ax1 = fig.add_subplot(gs[0, 1], sharey=ax0)

balmer_decrement_to_plot_corrected = []

for i in range(len(alpha_flux_corrected)):
    uncertainty = FL.BalmerDecrementUncertainty(alpha_flux_corrected[i], alpha_flux_err_corrected[i], beta_flux_corrected[i], beta_flux_err_corrected[i])
    if uncertainty / balmer_decrement_corrected[i] > 0.25:
        continue
    if balmer_decrement_corrected[i] > 5 or redshift_list[i] < 4 or redshift_list[i] > 7.5:
        continue

    # Append the valid data point to our list for the histogram
    balmer_decrement_to_plot_corrected.append((balmer_decrement_corrected[i]-2.86)/uncertainty)

    ax0.scatter(redshift_list[i], (balmer_decrement_corrected[i]-2.86)/uncertainty, s=10, alpha=0.5, c='g')

ax1.hist(balmer_decrement_to_plot_corrected, bins=100, orientation='horizontal', color='k', alpha=0.7,density=True)
#plot a gaussian distribution curve on ax1
from scipy.stats import norm
import numpy as np
mean=0
std_dev=1
x = np.linspace(-5, 5, 1000)
y = norm.pdf(x, mean, std_dev)
#scale y to match the histogram

ax1.plot(y, x, color='r', linestyle='--')

ax0.set_xlabel('Redshift')
ax0.set_ylabel('Corrected Balmer Decrement')
ax0.set_ylim(-5, 5)


In [ ]:
fig, ax = plt.subplots(figsize=(25,14))

ax.hist(balmer_decrement_to_plot_corrected, bins=100,  color='k', alpha=0.7,density=True)
#plot a gaussian distribution curve on ax1
from scipy.stats import norm
import numpy as np
mean=0
std_dev=1
x = np.linspace(-5, 5, 1000)
y = norm.pdf(x, mean, std_dev)
#scale y to match the histogram

ax.plot(x,y, color='r', linestyle='--', linewidth=2, label='Gaussian')
ax.set_xlim(-5, 5)

ax.spines['bottom'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['bottom'].set_color('black')
ax.spines['top'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['right'].set_color('black')
ax.yaxis.set_ticks_position('both')
ax.xaxis.set_ticks_position('both')
ax.xaxis.set_tick_params(width=2, direction='in', which='both', labelsize=20)
ax.yaxis.set_tick_params(width=2, direction='in', which='both', labelsize=20)
ax.xaxis.set_tick_params(length=6, which='major')
ax.xaxis.set_tick_params(length=3, which='minor')
ax.yaxis.set_tick_params(length=6, which='major')
ax.yaxis.set_tick_params(length=3, which='minor')
ax.grid(visible=True, which='major', color='gray', linestyle='--', linewidth=1)
ax.grid(visible=True, which='minor', color='lightgray', linestyle='--', linewidth=0.5)
ax.legend(fontsize=20, loc='best')
ax.set_title('Slit Loss Corrected Balmer Decrement Significance Distribution', fontsize=25)
ax.set_xlabel('(Corrected Balmer Decrement - 2.86) / Uncertainty', fontsize=20)
ax.set_ylabel('Density', fontsize=20)
plt.savefig('balmer_decrement_significance_distribution.pdf')

In [ ]:

ax.hist(balmer_decrement_to_plot_corrected, bins=100,  color='k', alpha=0.7,density=True)
#plot a gaussian distribution curve on ax1
from scipy.stats import norm
import numpy as np
mean=0
std_dev=1
x = np.linspace(-5, 5, 1000)
y = norm.pdf(x, mean, std_dev)
#scale y to match the histogram

ax.plot(x,y, color='r', linestyle='--', linewidth=2, label='Gaussian')
ax.set_xlim(-5, 5)

ax.spines['bottom'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['bottom'].set_color('black')
ax.spines['top'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['right'].set_color('black')
ax.yaxis.set_ticks_position('both')
ax.xaxis.set_ticks_position('both')
ax.xaxis.set_tick_params(width=2, direction='in', which='both', labelsize=20)
ax.yaxis.set_tick_params(width=2, direction='in', which='both', labelsize=20)
ax.xaxis.set_tick_params(length=6, which='major')
ax.xaxis.set_tick_params(length=3, which='minor')
ax.yaxis.set_tick_params(length=6, which='major')
ax.yaxis.set_tick_params(length=3, which='minor')
ax.grid(visible=True, which='major', color='gray', linestyle='--', linewidth=1)
ax.grid(visible=True, which='minor', color='lightgray', linestyle='--', linewidth=0.5)
ax.legend(fontsize=20, loc='best')
ax.set_title('Slit Loss Corrected Balmer Decrement Significance Distribution', fontsize=25)
ax.set_xlabel('(Corrected Balmer Decrement - 2.86) / Uncertainty', fontsize=20)
ax.set_ylabel('Density', fontsize=20)
plt.savefig('balmer_decrement_significance_distribution.pdf')